# 03 - Task A: Exploratory analysis

The four A1 metrics from the brief:
1. Monthly GMV by city and category
2. Monthly orders and unique active customers
3. Monthly repeat purchase rate (cumulative ≥2 delivered orders / active that month)
4. Delayed-order share by city and carrier

Reads the parquets produced by `02_clean_features.ipynb`.

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

DATA = Path('..') / 'data' / 'processed'

orders_e   = pd.read_parquet(DATA / 'orders_enriched.parquet')
items_e    = pd.read_parquet(DATA / 'items_enriched.parquet')
ships_e    = pd.read_parquet(DATA / 'shipments_enriched.parquet')
first_ord  = pd.read_parquet(DATA / 'customer_first_order.parquet')
products   = pd.read_csv(Path('..').resolve().parent / 'products.csv')

orders_e.shape, items_e.shape, ships_e.shape, first_ord.shape

((100000, 30), (169929, 11), (91994, 13), (23443, 6))

## A1.1 Monthly GMV by city and category

Item-level GMV joined to product category and the order's customer city. Months pivoted on the column axis so trends are easy to read.

In [8]:
items_full = (items_e
    .merge(orders_e[['order_id', 'city', 'order_month']], on='order_id', how='left')
    .merge(products[['product_id', 'category']], on='product_id', how='left')
)

gmv_city_cat_month = (items_full
    .groupby(['order_month', 'city', 'category'], as_index=False)['gmv_line'].sum()
    .rename(columns={'gmv_line': 'gmv'})
)

print('rows :', len(gmv_city_cat_month))
print('total gmv:', round(gmv_city_cat_month['gmv'].sum(), 2))
gmv_city_cat_month.head()

rows : 1080
total gmv: 3391994349.0


,order_month,city,category,gmv
0,2024-07,Ahmedabad,Books,110449.0
1,2024-07,Ahmedabad,Electronics,6231837.0
2,2024-07,Ahmedabad,Fashion,672491.0
3,2024-07,Ahmedabad,Grocery,117559.0
4,2024-07,Ahmedabad,Home & Kitchen,1341740.0


In [9]:
gmv_city_month = gmv_city_cat_month.groupby(['order_month', 'city'], as_index=False)['gmv'].sum()

fig = px.line(gmv_city_month.sort_values('order_month'),
              x='order_month', y='gmv', color='city',
              title='Monthly GMV by city')
fig.update_layout(height=450, legend_title_text='')
fig.show()

In [10]:
gmv_cat_month = gmv_city_cat_month.groupby(['order_month', 'category'], as_index=False)['gmv'].sum()

fig = px.line(gmv_cat_month.sort_values('order_month'),
              x='order_month', y='gmv', color='category',
              title='Monthly GMV by category')
fig.update_layout(height=450, legend_title_text='')
fig.show()

In [11]:
# Compact pivot: city x month, total GMV in millions
pivot_city = (gmv_city_month
    .pivot(index='city', columns='order_month', values='gmv')
    .div(1_000_000).round(2))

pivot_city['total'] = pivot_city.sum(axis=1).round(2)
pivot_city.sort_values('total', ascending=False)

order_month,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,total
city,,,,,,,,,,,,,,,,,,,
Mumbai,32.77,33.98,32.09,33.78,31.07,33.94,32.91,32.80,31.86,33.06,34.16,33.18,35.74,34.23,33.41,29.84,36.68,34.23,599.73
Delhi,29.97,30.51,29.65,27.11,30.04,27.28,28.44,28.14,29.48,29.24,31.45,32.21,31.28,32.77,28.12,29.98,30.96,28.73,535.36
Bangalore,27.64,24.95,28.91,27.51,26.83,28.06,24.92,24.37,28.17,26.32,27.39,27.32,27.30,27.53,27.36,30.59,26.41,24.79,486.37
Hyderabad,20.42,20.67,18.88,19.66,18.35,20.55,21.75,17.94,21.01,18.38,16.66,16.78,18.96,21.67,19.54,20.97,19.94,19.05,351.18
Chennai,19.17,18.14,17.34,20.01,17.59,15.85,18.46,16.55,15.34,15.41,14.15,17.11,15.65,16.62,17.87,18.58,18.75,13.91,306.50
Pune,14.25,17.29,14.41,14.08,12.75,13.97,15.75,12.86,14.95,13.83,14.48,16.22,14.42,15.46,14.67,14.52,15.51,16.03,265.45
Kolkata,15.49,11.96,13.97,13.47,12.87,12.79,12.19,11.63,14.50,10.82,13.27,12.34,12.63,13.02,12.51,12.91,12.50,12.42,231.29
Ahmedabad,8.47,11.53,9.30,10.14,8.24,9.36,11.09,10.89,8.64,9.89,9.95,9.38,9.28,9.69,8.24,10.41,8.72,9.36,172.58
Jaipur,6.93,8.30,7.06,6.64,8.53,9.17,7.75,5.79,8.73,6.19,9.68,7.67,7.49,7.48,7.01,8.13,6.51,7.59,136.65


## A1.2 Monthly orders and active customers

In [12]:
orders_month = (orders_e.groupby('order_month').agg(
    orders            = ('order_id', 'count'),
    active_customers  = ('customer_id', 'nunique'),
    gmv               = ('order_gmv', 'sum'),
).reset_index())

orders_month['orders_per_customer'] = (orders_month['orders'] / orders_month['active_customers']).round(3)
orders_month

,order_month,orders,active_customers,gmv,orders_per_customer
0,2024-07,5634,4947,194561564.0,1.139
1,2024-08,5551,4890,194832277.0,1.135
2,2024-09,5595,4930,186364791.0,1.135
3,2024-10,5791,5102,190639228.0,1.135
4,2024-11,5504,4849,184720074.0,1.135
5,2024-12,5719,4952,186549751.0,1.155
6,2025-01,5604,4898,189911685.0,1.144
7,2025-02,5163,4581,176686798.0,1.127
8,2025-03,5627,4964,192612052.0,1.134
9,2025-04,5515,4866,180771429.0,1.133


In [13]:
fig = px.line(orders_month, x='order_month', y=['orders', 'active_customers'],
              title='Monthly orders and active customers')
fig.update_layout(height=400, legend_title_text='')
fig.show()

## A1.3 Monthly repeat purchase rate

Definition: customers who placed an order in month M *and* have ≥2 delivered orders by the end of M, divided by customers active in M.

A customer becomes "repeat" the moment their second delivered order arrives. We capture that timestamp once, then for each month count active customers who already crossed that threshold.

In [14]:
delivered = orders_e[orders_e['delivered_at'].notna()].copy()
delivered = delivered.sort_values(['customer_id', 'delivered_at'])
delivered['n_so_far'] = delivered.groupby('customer_id').cumcount() + 1

# When did each customer first qualify as repeat (2nd delivery)?
became_repeat_at = (delivered[delivered['n_so_far'] == 2]
                    .set_index('customer_id')['delivered_at'])

months = sorted(orders_e['order_month'].dropna().unique())
rows = []
for m in months:
    end_of_m = pd.Period(m, freq='M').to_timestamp(how='end')
    active_set = set(orders_e.loc[orders_e['order_month'] == m, 'customer_id'])
    repeat_set = set(became_repeat_at[became_repeat_at <= end_of_m].index) & active_set
    rows.append({
        'order_month': m,
        'active_customers': len(active_set),
        'repeat_customers_active': len(repeat_set),
        'repeat_rate': round(len(repeat_set) / len(active_set), 4) if active_set else 0,
    })

repeat_monthly = pd.DataFrame(rows)
repeat_monthly

,order_month,active_customers,repeat_customers_active,repeat_rate
0,2024-07,4947,428,0.0865
1,2024-08,4890,1155,0.2362
2,2024-09,4930,1787,0.3625
3,2024-10,5102,2318,0.4543
4,2024-11,4849,2614,0.5391
5,2024-12,4952,3069,0.6197
6,2025-01,4898,3315,0.6768
7,2025-02,4581,3306,0.7217
8,2025-03,4964,3735,0.7524
9,2025-04,4866,3858,0.7928


In [15]:
fig = px.line(repeat_monthly, x='order_month', y='repeat_rate',
              title='Monthly repeat purchase rate', markers=True)
fig.update_layout(height=400, yaxis_tickformat='.0%')
fig.show()

## A1.4 Delayed-order share by city and carrier

Denominator is shipments where the outcome is known (excludes InTransit). Numerator is anything not OnTime (treats Lost as a failed promise).

In [16]:
# Use ship_to_city (logistics view) since the brief frames this as a logistics question
evaluable = ships_e[ships_e['is_delayed'].notna()].copy()
evaluable['is_delayed'] = evaluable['is_delayed'].astype(bool)

delay_city = (evaluable.groupby('ship_to_city')
    .agg(shipments=('shipment_id', 'count'),
         delayed=('is_delayed', 'sum'))
    .assign(delayed_rate=lambda d: (d['delayed'] / d['shipments']).round(4))
    .sort_values('delayed_rate', ascending=False)
    .reset_index())

delay_carrier = (evaluable.groupby('carrier')
    .agg(shipments=('shipment_id', 'count'),
         delayed=('is_delayed', 'sum'))
    .assign(delayed_rate=lambda d: (d['delayed'] / d['shipments']).round(4))
    .sort_values('delayed_rate', ascending=False)
    .reset_index())

print('By ship_to_city:')
print(delay_city.to_string(index=False))
print()
print('By carrier:')
print(delay_carrier.to_string(index=False))

By ship_to_city:
ship_to_city  shipments  delayed  delayed_rate
      Jaipur       3604     1873        0.5197
     Lucknow       3468     1785        0.5147
     Kolkata       6100     2678        0.4390
   Ahmedabad       4245     1199        0.2824
     Chennai       7882     2206        0.2799
        Pune       6834     1913        0.2799
  Chandigarh       2701      753        0.2788
       Kochi       1686      445        0.2639
   Bangalore      12313     2589        0.2103
       Delhi      13770     2888        0.2097
   Hyderabad       8850     1846        0.2086
      Mumbai      15498     3094        0.1996

By carrier:
  carrier  shipments  delayed  delayed_rate
Delhivery      27918     9364        0.3354
    Ekart      20670     6772        0.3276
 BlueDart      20741     5755        0.2775
  InHouse      17622     1378        0.0782


In [17]:
delay_cc = (evaluable.groupby(['ship_to_city', 'carrier'])
    .agg(shipments=('shipment_id', 'count'),
         delayed=('is_delayed', 'sum'))
    .assign(delayed_rate=lambda d: d['delayed'] / d['shipments'])
    .reset_index())

heatmap = delay_cc.pivot(index='ship_to_city', columns='carrier', values='delayed_rate')

fig = px.imshow(heatmap.round(3), text_auto=True, aspect='auto',
                color_continuous_scale='Reds',
                title='Delayed-order rate by ship-to city and carrier')
fig.update_layout(height=550)
fig.show()

In [18]:
# Trend: is the on-time rate actually declining as the brief suggests?
evaluable['shipped_month'] = evaluable['shipped_at'].dt.to_period('M').astype(str)

trend = (evaluable.groupby('shipped_month')
    .agg(shipments=('shipment_id', 'count'),
         delayed=('is_delayed', 'sum'))
    .assign(delayed_rate=lambda d: d['delayed'] / d['shipments'])
    .reset_index())

fig = px.line(trend, x='shipped_month', y='delayed_rate', markers=True,
              title='Monthly delayed-order rate (all carriers, all cities)')
fig.update_layout(height=400, yaxis_tickformat='.0%')
fig.show()

trend

,shipped_month,shipments,delayed,delayed_rate
0,2024-07,4863,979,0.201316
1,2024-08,4823,999,0.207132
2,2024-09,4853,1015,0.209149
3,2024-10,5013,2441,0.486934
4,2024-11,4794,2335,0.487067
5,2024-12,4939,1036,0.209759
6,2025-01,4888,986,0.201718
7,2025-02,4479,896,0.200045
8,2025-03,4927,1025,0.208037
9,2025-04,4800,947,0.197292


Numbers from the four blocks above feed into the A2 write-up (next notebook step) and the Streamlit app. Stop here for review.